In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")

In [2]:
if(gemini_api_key is None):
    print("Please set the GOOGLE_API_KEY environment variable.")
else:
    print("GOOGLE_API_KEY is set.")

GOOGLE_API_KEY is set.


In [65]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core import Settings
from llama_index.core import StorageContext, load_index_from_storage
from IPython.display import Markdown, display
from llama_index.llms.gemini import Gemini
# from google import genai
# from google.genai import types
import google.generativeai as genai
# from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [23]:
genai.configure(api_key=gemini_api_key)
# client = genai.Client(api_key='gemini_api_key')

In [24]:
# for models in genai.list_models():
#     if 'generateContent' in models.supported_generation_methods:
#         print(models.name)

In [56]:
# Data Loading

raw_documents = SimpleDirectoryReader(input_files=["./data/MLDOC.txt"]).load_data()

In [39]:
raw_documents[0]

Document(id_='2a9d1604-f2a1-4057-8817-f413cb513895', embedding=None, metadata={'file_path': 'data\\MLDOC.txt', 'file_name': 'MLDOC.txt', 'file_type': 'text/plain', 'file_size': 21968, 'creation_date': '2026-03-10', 'last_modified_date': '2026-03-10'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='What is machine learning?\nMachine learning is a branch of artificial intelligence (AI) and computer science which\nfocuses on the use of data and algorithms to imitate the way that humans learn,\ngradually improving its accuracy.\nIBM has a rich history with machine learning. One of its own, Arthur Samuel, is credited\nfor coining the term, “ma

In [72]:
embedding = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

2026-03-10 15:24:03,901 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-03-10 15:24:04,365 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-10 15:24:04,416 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-03-10 15:24:04,852 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-10 15:24:04,906 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-10 15:24:05,341 - INFO - HTTP Request: HEAD https://huggingface.co/sent

In [83]:
gemini_model = Gemini(model_name="models/gemini-2.5-flash")

C:\Users\Manas tiwari\AppData\Local\Temp\ipykernel_53876\4275832539.py:1: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  gemini_model = Gemini(model_name="models/gemini-2.5-flash")


In [84]:
Settings.llm = gemini_model
Settings.embed_model = embedding
Settings.chunk_size = 512
Settings.chunk_overlap = 40

In [85]:
index = VectorStoreIndex.from_documents(raw_documents)

In [77]:
index

In [86]:
index.storage_context.persist(persist_dir="./storage")

In [79]:
# what is as_chat_engine in the context of llama_index?
# as_chat_engine is a method that converts a query engine into a chat engine. A chat engine is a type of query engine that is designed to handle multi-turn conversations. It can maintain context across multiple queries and responses, allowing for more natural and coherent interactions with the user.

#  what is as_query_engine in the context of llama_index?
# as_query_engine is a method that converts a chat engine into a query engine. A query engine is a type of chat engine that is designed to handle single-turn queries. It does not maintain context across multiple queries and responses, and is typically used for simple question-answering tasks.

# what is the difference between a query engine and a chat engine in the context of llama_index?
# The main difference between a query engine and a chat engine in the context of llama_index is that a query engine is designed to handle single-turn queries, while a chat engine is designed to handle multi-turn conversations. A query engine does not maintain context across multiple queries and responses, while a chat engine can maintain context, allowing for more natural and coherent interactions with the user.

In [87]:
query_engine = index.as_query_engine()

In [92]:
response = query_engine.query("What is machine learning?")

In [93]:
response.response

'Machine learning is a branch of artificial intelligence and computer science that focuses on using data and algorithms to imitate how humans learn, progressively enhancing its accuracy.'

In [94]:
response = query_engine.query("What are the applications of machine learning?")

In [95]:
print(response.response)

Machine learning is applied in various real-world scenarios, including:

*   **Speech recognition:** This capability translates human speech into a written format and is used in mobile devices for voice search or to improve accessibility for texting.
*   **Customer service:** Online chatbots powered by machine learning replace human agents, answering frequently asked questions and providing personalized advice on websites and social media platforms.
*   **Computer vision:** This technology allows computers to derive meaningful information from digital images, videos, and other visual inputs, with applications in photo tagging, radiology imaging, and self-driving cars.
*   **Recommendation engines:** These systems use past consumption behavior data to discover trends and make relevant product recommendations to customers, as seen in online retail.
*   **Robotic process automation (RPA):** Also known as software robotics, RPA uses intelligent automation to perform repetitive manual tasks

In [96]:
response = query_engine.query("What is capital of India?")

In [97]:
print(response.response)

The requested information is not available in the provided context.


In [98]:
response = query_engine.query("What is the difference between machine learning and deep learning?")

In [99]:
print(response.response)

The primary distinction between machine learning and deep learning lies in their learning approaches. Deep learning algorithms can process unstructured data in its raw form, such as text or images, and automatically identify distinguishing features, which minimizes the need for human involvement. This characteristic allows deep learning to scale effectively with large datasets.

In contrast, classical machine learning relies more heavily on human intervention. Experts are typically required to define the features that help the algorithm understand differences in data inputs, and it generally necessitates more structured data for learning.

Furthermore, the term "deep" in deep learning refers to the architecture of its neural networks. A deep learning algorithm utilizes a neural network with more than three layers, including the input and output layers. A basic neural network, on the other hand, typically consists of only three layers.
